# MRI Dementia Retraining - MobileNetV2, 3 Classes

This notebook trains a MobileNetV2-based classifier on the combined Kaggle/OASIS dataset, but maps the original 4 folders into 3 target classes:

```python
['NonDemented', 'MildOrVeryMildDemented', 'ModerateDemented']
```

Folder mapping:

```python
NonDemented       -> NonDemented
VeryMildDemented  -> MildOrVeryMildDemented
MildDemented      -> MildOrVeryMildDemented
ModerateDemented  -> ModerateDemented
```

Important: this model uses `tf.keras.applications.mobilenet_v2.preprocess_input`, so do **not** also divide images by `255`.


In [ ]:
import os
import zipfile
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)

## Upload Or Mount Dataset

Option A: upload a zip named `combined_dataset_plus_oasis_axial_coronal_180_shuffled.zip` to Colab.

Option B: mount Google Drive and point `DATASET_DIR` to the extracted folder.

In [ ]:
# Optional: mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
ZIP_PATH = "/content/combined_dataset_plus_oasis_axial_coronal_180_shuffled.zip"
EXTRACT_DIR = "/content/mri_dataset"

if os.path.exists(ZIP_PATH):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extracted to:", EXTRACT_DIR)
else:
    print("Zip not found. If using Google Drive, set DATASET_DIR manually below.")

In [ ]:
# Change this if your extracted folder name/path is different.
# This default points to the newest combined dataset with Disc1 + Disc2 OASIS.
DATASET_DIR = "/content/mri_dataset/combined_dataset_plus_oasis_disc1_disc2_axial_coronal_180_shuffled"

# Google Drive example:
# DATASET_DIR = "/content/drive/MyDrive/combined_dataset_plus_oasis_disc1_disc2_axial_coronal_180_shuffled"

source_class_names = [
    "MildDemented",
    "ModerateDemented",
    "NonDemented",
    "VeryMildDemented",
]

class_names = [
    "NonDemented",
    "MildOrVeryMildDemented",
    "ModerateDemented",
]

source_to_target_class = {
    "NonDemented": "NonDemented",
    "VeryMildDemented": "MildOrVeryMildDemented",
    "MildDemented": "MildOrVeryMildDemented",
    "ModerateDemented": "ModerateDemented",
}

target_class_to_index = {name: i for i, name in enumerate(class_names)}

dataset_path = Path(DATASET_DIR)
print("Dataset path exists:", dataset_path.exists())
print("Target class order:", class_names)
print("Source folder counts:")

for cls in source_class_names:
    class_dir = dataset_path / cls
    print(cls, len(list(class_dir.glob("*.png"))) if class_dir.exists() else "missing")


## Collect Image Paths

In [ ]:
image_paths = []
labels = []
source_labels = []

for source_cls in source_class_names:
    class_dir = dataset_path / source_cls
    target_cls = source_to_target_class[source_cls]
    target_idx = target_class_to_index[target_cls]

    for p in class_dir.glob("*"):
        if p.suffix.lower() in [".png", ".jpg", ".jpeg"] and not p.name.startswith("._"):
            image_paths.append(str(p))
            labels.append(target_idx)
            source_labels.append(source_cls)

image_paths = np.array(image_paths)
labels = np.array(labels)
source_labels = np.array(source_labels)

print("Total images:", len(image_paths))
print("3-class target order:", class_names)

print("Target class counts:")
for i, cls in enumerate(class_names):
    print(cls, int(np.sum(labels == i)))

print("\nOriginal source folder counts included:")
for cls in source_class_names:
    print(cls, int(np.sum(source_labels == cls)))


## Train / Validation Split

This is a stratified image-level split. For a stricter medical-imaging experiment, use the `manifest.csv` to split by OASIS subject/source.

In [ ]:
train_paths, val_paths, y_train, y_val = train_test_split(
    image_paths,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print("Train:", len(train_paths))
print("Validation:", len(val_paths))

for i, cls in enumerate(class_names):
    print(cls, "train:", int(np.sum(y_train == i)), "val:", int(np.sum(y_val == i)))

## Class Weights

In [ ]:
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=y_train
)

class_weights = dict(enumerate(class_weights_array))
print(class_weights)

## Build `tf.data` Datasets

Images are loaded as RGB because MobileNetV2 was pretrained on RGB ImageNet images.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img, label


train_ds = tf.data.Dataset.from_tensor_slices((train_paths, y_train))
train_ds = train_ds.shuffle(buffer_size=len(train_paths), seed=42)
train_ds = train_ds.map(load_image, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, y_val))
val_ds = val_ds.map(load_image, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

## Data Augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.08),
    tf.keras.layers.RandomContrast(0.10),
])

## Build MobileNetV2 Model

MobileNetV2 expects `preprocess_input`, which maps images into its expected range. Do not add `Rescaling(1./255)` with this setup.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.35)(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Train Classifier Head

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_mobilenet_mri_3class.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=6,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks
)

## Fine-Tune Top MobileNet Layers

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)

## Evaluate Best Model

In [ ]:
best_model = tf.keras.models.load_model("best_mobilenet_mri_3class.keras")

val_loss, val_acc = best_model.evaluate(val_ds)
print("Validation loss:", val_loss)
print("Validation accuracy:", val_acc)

In [ ]:
y_true = []
y_pred = []

for x_batch, y_batch in val_ds:
    preds = best_model.predict(x_batch, verbose=0)
    y_true.extend(y_batch.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

print("Classification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))

## Plot Training Curves

In [ ]:
def plot_history(histories):
    acc = []
    val_acc = []
    loss = []
    val_loss = []

    for h in histories:
        acc += h.history.get("accuracy", [])
        val_acc += h.history.get("val_accuracy", [])
        loss += h.history.get("loss", [])
        val_loss += h.history.get("val_loss", [])

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(acc, label="train_acc")
    plt.plot(val_acc, label="val_acc")
    plt.legend()
    plt.title("Accuracy")

    plt.subplot(1, 2, 2)
    plt.plot(loss, label="train_loss")
    plt.plot(val_loss, label="val_loss")
    plt.legend()
    plt.title("Loss")

    plt.show()

plot_history([history, history_finetune])

## Save Final Model

In [ ]:
best_model.save("mobilenet_mri_dementia_3class_final.keras")
print("Saved: mobilenet_mri_dementia_3class_final.keras")